# Stage 2 — Full CTLS Objective

**Goal:** Train with the full CTLS objective (`ℒ_task + λ·ℒ_cons`) and compare
the circuit latent space against the Stage 1 baseline.

**Core empirical test of the central claim:**
- Do circuit space clusters get tighter (higher silhouette)?
- Does task accuracy change significantly?
- Does the UMAP show visually cleaner separation?

**Training details:**
- λ is warmed up linearly from 0 → 1 over the first 10 epochs
- τ (soft mask temperature) is cosine-annealed from 1.0 → 0.1 over 80 epochs
- Data loader returns paired same-class batches (x₁, x₂, label)
- Single forward pass on `[x₁; x₂]` — no extra compute overhead

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({"Colab" if IN_COLAB else "local"})")  


In [ ]:
with open('configs/ctls.yaml') as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, default_flow_style=False))

## 1. Train CTLS Model

Expected training time: ~25–35 min on a single GPU (slightly longer than baseline
due to the paired forward pass, but not 2x — it's one forward pass per batch).

Watch the `cons` loss column: it should decrease steadily after the λ warmup ends
at epoch 10. If it doesn't move, check the learning rate and λ schedule.

**Skip if you have `experiments/ctls/best.pt`.**

In [ ]:
from training.trainer import Trainer

trainer = Trainer(config)
trainer.train()

## 2. Load Both Models for Side-by-Side Comparison

In [ ]:
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders

def load_model(config, checkpoint_path, device):
    mcfg = config['model']
    ecfg = mcfg['meta_encoder']
    tcfg = config['training']

    soft_mask = SoftMask(init_temperature=tcfg['temperature']['init']).to(device)
    backbone = CTLSBackbone(
        arch=mcfg['arch'],
        num_classes=mcfg['num_classes'],
        soft_mask=soft_mask,
        pretrained=False,
    ).to(device)
    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'mlp'),
    ).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    backbone.eval()
    meta_encoder.eval()
    return backbone, meta_encoder, ckpt

# Load CTLS model
backbone_ctls, meta_enc_ctls, ckpt_ctls = load_model(
    config, 'experiments/ctls/best.pt', DEVICE
)
print(f"CTLS:     epoch={ckpt_ctls['epoch']}, val_acc={ckpt_ctls['val_acc']:.4f}")

# Load baseline model for comparison
with open('configs/baseline.yaml') as f:
    baseline_config = yaml.safe_load(f)
backbone_base, meta_enc_base, ckpt_base = load_model(
    baseline_config, 'experiments/baseline/best.pt', DEVICE
)
print(f"Baseline: epoch={ckpt_base['epoch']}, val_acc={ckpt_base['val_acc']:.4f}")

In [ ]:
dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'],
    batch_size=dcfg['batch_size'],
    num_workers=dcfg.get('num_workers', 4),
    augment=False,
    download=True,
)

## 3. UMAP: CTLS vs Baseline Side-by-Side

Four panels: [CTLS circuit | CTLS output] and [Baseline circuit | Baseline output].
The key comparison is CTLS circuit space vs Baseline circuit space.

In [ ]:
from evaluation.circuit_viz import CircuitVisualizer

viz_ctls = CircuitVisualizer(backbone_ctls, meta_enc_ctls, val_loader, DEVICE)
viz_base = CircuitVisualizer(backbone_base, meta_enc_base, val_loader, DEVICE)

print('Computing CTLS UMAP...')
fig_ctls = viz_ctls.plot_umap(
    title='Stage 2 — CTLS',
    compare_output=True,
    max_samples=5000,
)
os.makedirs('experiments/ctls', exist_ok=True)
fig_ctls.savefig('experiments/ctls/umap_ctls.png', dpi=150, bbox_inches='tight')
plt.show()

print('Computing Baseline UMAP...')
fig_base = viz_base.plot_umap(
    title='Stage 1 — Baseline',
    compare_output=True,
    max_samples=5000,
)
fig_base.savefig('experiments/baseline/umap_baseline_s2.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Quantitative Comparison: Silhouette Scores

In [ ]:
print('Measuring cluster separation...')
scores_ctls = viz_ctls.cluster_separation_score(max_samples=5000)
scores_base = viz_base.cluster_separation_score(max_samples=5000)

print()
print('=== Silhouette Score Comparison ===')
print(f"{'Metric':<35} {'Baseline':>10} {'CTLS':>10} {'Delta':>10}")
print('-' * 65)
print(f"{'Circuit space silhouette':<35} "
      f"{scores_base['silhouette_circuit']:>10.4f} "
      f"{scores_ctls['silhouette_circuit']:>10.4f} "
      f"{scores_ctls['silhouette_circuit'] - scores_base['silhouette_circuit']:>+10.4f}")
print(f"{'Output space silhouette':<35} "
      f"{scores_base['silhouette_output']:>10.4f} "
      f"{scores_ctls['silhouette_output']:>10.4f} "
      f"{scores_ctls['silhouette_output'] - scores_base['silhouette_output']:>+10.4f}")
print(f"{'Val accuracy':<35} "
      f"{ckpt_base['val_acc']:>10.4f} "
      f"{ckpt_ctls['val_acc']:>10.4f} "
      f"{ckpt_ctls['val_acc'] - ckpt_base['val_acc']:>+10.4f}")

## 5. Per-Layer Silhouette: Where Does CTLS Improve Structure?

The depth-weighted loss should push improvement toward later layers.
This plot tells us if that's actually happening.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import torch.nn.functional as F

def collect_layer_acts(backbone, loader, n=2000):
    all_layers = [[] for _ in range(len(backbone.layer_dims))]
    all_labels = []
    total = 0
    with torch.no_grad():
        for x, labels in loader:
            x = x.to(DEVICE)
            _, traj = backbone(x)
            for i, h in enumerate(traj):
                all_layers[i].append(h.cpu())
            all_labels.append(labels)
            total += x.shape[0]
            if total >= n:
                break
    layers = [torch.cat(l, 0)[:n].numpy() for l in all_layers]
    labels = torch.cat(all_labels, 0)[:n].numpy()
    return layers, labels

def layer_silhouettes(layers, labels):
    scores = []
    for acts in layers:
        a = PCA(n_components=min(50, acts.shape[1])).fit_transform(acts) if acts.shape[1] > 50 else acts
        s = silhouette_score(a, labels, metric='euclidean', sample_size=min(1000, len(labels)))
        scores.append(s)
    return scores

print('Collecting CTLS activations...')
layers_ctls, labels_ctls = collect_layer_acts(backbone_ctls, val_loader)
sil_ctls = layer_silhouettes(layers_ctls, labels_ctls)

print('Collecting Baseline activations...')
layers_base, labels_base = collect_layer_acts(backbone_base, val_loader)
sil_base = layer_silhouettes(layers_base, labels_base)

print('Done.')

In [ ]:
x = range(1, len(sil_ctls) + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, sil_base, 'o--', color='steelblue', label='Baseline', lw=2)
ax.plot(x, sil_ctls, 's-', color='darkorange', label='CTLS', lw=2)
ax.fill_between(x, sil_base, sil_ctls,
                where=[c > b for c, b in zip(sil_ctls, sil_base)],
                alpha=0.2, color='darkorange', label='CTLS improvement')
ax.set_xlabel('Layer index')
ax.set_ylabel('Silhouette score')
ax.set_title('Per-layer semantic structure: CTLS vs Baseline')
ax.axhline(0, color='gray', linestyle='--', lw=0.8)
ax.set_xticks(list(x))
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/ctls/per_layer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. λ and τ Schedule Sanity Check

Verify the schedules behave as intended before inspecting training curves.

In [ ]:
from training.schedulers import LambdaScheduler, TauScheduler

tcfg = config['training']
lambda_sched = LambdaScheduler(
    init_val=tcfg['lambda_consistency']['init'],
    final_val=tcfg['lambda_consistency']['final'],
    warmup_epochs=tcfg['lambda_consistency']['warmup_epochs'],
)
tau_sched = TauScheduler(
    init_val=tcfg['temperature']['init'],
    final_val=tcfg['temperature']['final'],
    anneal_epochs=tcfg['temperature']['anneal_epochs'],
)

epochs = range(tcfg['epochs'])
lambdas = [lambda_sched.get(e) for e in epochs]
taus = [tau_sched.get(e) for e in epochs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(list(epochs), lambdas, lw=2, color='steelblue')
ax1.set(xlabel='Epoch', ylabel='λ', title='Consistency loss weight (λ schedule)')
ax1.grid(True, alpha=0.3)
ax2.plot(list(epochs), taus, lw=2, color='darkorange')
ax2.set(xlabel='Epoch', ylabel='τ', title='Soft mask temperature (τ schedule)')
ax2.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/ctls/schedules.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Fill in the comparison table:

| Metric | Stage 1 Baseline | Stage 2 CTLS | Delta |
|--------|-----------------|-------------|-------|
| Val accuracy | ___ | ___ | ___ |
| Circuit space silhouette | ___ | ___ | ___ |
| Output space silhouette | ___ | ___ | ___ |
| Peak per-layer silhouette | ___ (layer ___) | ___ (layer ___) | ___ |

**Interpretation guide:**
- Circuit silhouette increase with minimal accuracy drop → consistency loss is working
- Output silhouette change should be small → circuit space carries distinct information
- Per-layer improvement concentrated in later layers → depth-weighting is effective

**Next:** Run `stage3_embedding_compare.ipynb` to verify circuit ≠ output space.